# AV2 · Testes de contexto e metamórficos
**Crawl4AI + Qwen3-4B + QA Gate da Pessoa 3 + prompts P1/P2/P3 do grupo**

Rode as células **de cima para baixo**, uma por vez (▶ ou Shift+Enter).
Antes: menu *Ambiente de execução → Alterar tipo de ambiente de execução → **GPU T4***.

In [ ]:
!nvidia-smi | head -12

## 1. Instalar Crawl4AI (≈ 3–5 min)

In [ ]:
!pip install -q crawl4ai pydantic
!crawl4ai-setup
!crawl4ai-doctor

## 2. Instalar Ollama e baixar o Qwen3-4B (≈ 3–5 min)

In [ ]:
!apt-get install -y -qq pciutils zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(8)
!ollama pull qwen3:4b
# aquece o modelo e o mantém na GPU por 2 h (evita recarregar a cada chamada)
!curl -s http://localhost:11434/api/generate -d '{"model":"qwen3:4b","prompt":"Diga apenas: ok","stream":false,"keep_alive":"2h"}' > /dev/null
!ollama ps

> Na coluna **PROCESSOR** do `ollama ps`, o ideal é aparecer **100% GPU**. Se aparecer CPU, o teste fica muito lento: confira se a GPU T4 está ligada (célula 0).

## 3. Enviar o pacote
Clique em *Escolher arquivos* e selecione **`QS_testes.zip`**.

In [ ]:
from google.colab import files
files.upload()
!unzip -oq QS_testes.zip
%cd QS_testes/5_TestesContextoMetamorfico
!ls

## 4. Treino offline (sem LLM, 10 segundos)
Confere se tudo está no lugar. As saídas são **fictícias**: não use como evidência.

In [ ]:
!python run_testes.py --modo offline

## 5. Testes reais

`/no_think` desliga o "raciocínio" do Qwen3 (senão ele escreve o pensamento antes do JSON).
Cada rodada grava as saídas **na hora**: se o Colab cair, use `--retomar resultados/<pasta>` para continuar de onde parou.

⏱ Cada chamada leva de ~1 a 2 min. Rodada do **P2** = 21 chamadas; **P1** e **P3** = 15 chamadas cada. Se faltar tempo, troque `--repeticoes 3` por `--repeticoes 1`.

### 5a. Prompt P2 (roda também CT-MET-02b e 02c, derivados do P2)

In [ ]:
!python run_testes.py --modo online --provider ollama/qwen3:4b --base-url http://localhost:11434 \
    --prompt p2 --repeticoes 3 --sufixo "/no_think"

### 5b. Prompt P1

In [ ]:
!python run_testes.py --modo online --provider ollama/qwen3:4b --base-url http://localhost:11434 \
    --prompt p1 --repeticoes 3 --sufixo "/no_think"

### 5c. Prompt P3 (few-shot)

In [ ]:
!python run_testes.py --modo online --provider ollama/qwen3:4b --base-url http://localhost:11434 \
    --prompt p3 --repeticoes 3 --sufixo "/no_think"

## 6. Comparação entre os prompts (tabela pronta para o relatório)

In [ ]:
!python consolidar.py

## 7. Ver o detalhe na tela

In [ ]:
import glob, pandas as pd
pd.set_option('display.max_colwidth', 100); pd.set_option('display.width', 250)
for f in sorted(glob.glob('resultados/*_online_*/evidencias.csv')):
    print('\n' + f)
    display(pd.read_csv(f)[['id_variante','prompt','repeticao','latencia_s','qa_valido','status','problemas','revisoes']])

## 8. Baixar tudo ⬇️ (faça isso antes de fechar o Colab!)
Os arquivos **somem** quando a sessão termina. O zip contém `evidencias.csv`, `resumo.md`, os JSONs brutos (`saidas/`), a quarentena do QA Gate e o consolidado.

In [ ]:
!zip -rq resultados_testes.zip resultados
from google.colab import files
files.download('resultados_testes.zip')

### (Opcional) Guardar uma cópia no Google Drive
Útil se a sessão for cair. Rode antes de qualquer risco de desconexão.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/QS_testes_resultados
!cp -r resultados/* /content/drive/MyDrive/QS_testes_resultados/

## Depois dos testes
1. Abra cada `evidencias.csv` e preencha **`categoria_falha`** e **`severidade`** nas linhas FALHOU/REVISAR.
2. Entregue o `resultados_testes.zip` à Pessoa 7 (métricas e planilha) e anexe os JSONs de `saidas/` ao repositório.
3. Registre no relatório: modelo `ollama/qwen3:4b`, versão do Crawl4AI (`!pip show crawl4ai`) e os prompts usados.